In [4]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Download Acta Pdf File from Website

In [ ]:
from pathlib import Path
from elecc_colombia.config import setup_logger
from elecc_colombia.config import (RAW_DATA_DIR, 
                                   INTERIM_DATA_DIR,
                                   PROCESSED_DATA_DIR,
                                   )

from elecc_colombia.config import (HEADLESS, 
                                   SLOW_MO, 
                                   BASE_URL,
                                   DEPARTMENT_ID, 
                                   READY_SELECTOR)

from playwright.async_api import async_playwright, Page

from elecc_colombia.browser_utils import (navigate, 
                                          select_first_option,
                                          get_options,
                                          select_option_by_text)

from elecc_colombia.actas_scraper import (click_consultar, 
                                          extract_mesas, 
                                          download_acta_direct,
                                          select_page_size,
                                          download_first_acta, 
                                          download_all_actas)
#from loguru import logger
#setup_logger()

### Download Acta PDF from First Mesa

In [3]:
async with async_playwright() as pw:
    browser = await pw.chromium.launch(headless=HEADLESS, slow_mo=SLOW_MO)
    page = await browser.new_page()
    url = BASE_URL + str(DEPARTMENT_ID)
    print(f"Navigating to URL: {url}")
    try:
        await navigate(page, url, READY_SELECTOR)

        print("\n── Municipio ──────────────────────────────────")
        municipio = await select_first_option(page, "seleccione el municipio")
        print(f"  ✓ Municipio selected: {municipio}")

        print("\n── Zona ───────────────────────────────────────")
        zona = await select_first_option(page, "seleccione la zona")
        print(f"  ✓ Zona selected: {zona}")

        print("\n── Puesto ─────────────────────────────────────")
        puesto = await select_first_option(page, "seleccione el puesto")
        print(f"  ✓ Puesto selected: {puesto}")

        await click_consultar(page)
        mesas = await extract_mesas(page)
        await download_first_acta(page, RAW_DATA_DIR)

        print(f"\nDone — {len(mesas)} mesa(s) extracted.")

    finally:
        await page.close()
        await browser.close()

Navigating to URL: https://divulgacione14presidente.registraduria.gov.co/departamento/72
Opening: https://divulgacione14presidente.registraduria.gov.co/departamento/72
Page loaded.

── Municipio ──────────────────────────────────
Clicking 'seleccione el municipio' container...
  Dropdown is open!
  Found 4 option(s):
    [0] 006 — CUMARIBO (100%)
    [1] 008 — LA PRIMAVERA (100%)
    [2] 001 — PUERTO CARREÑO (100%)
    [3] 002 — SANTA ROSALIA (100%)
  Selecting [0]: '006 — CUMARIBO (100%)'
  ✓ Municipio selected: 006 — CUMARIBO (100%)

── Zona ───────────────────────────────────────
Clicking 'seleccione la zona' container...
  Dropdown is open!
  Found 2 option(s):
    [0] Zona 00
    [1] Zona 99
  Selecting [0]: 'Zona 00'
  ✓ Zona selected: Zona 00

── Puesto ─────────────────────────────────────
Clicking 'seleccione el puesto' container...
  Dropdown is open!
  Found 1 option(s):
    [0] 00 - PUESTO CABECERA MUNICIPAL
  Selecting [0]: '00 - PUESTO CABECERA MUNICIPAL'
  ✓ Puesto selec

### Download Acta PDF from Multiple Mesas

In [22]:
async with async_playwright() as pw:
    browser = await pw.chromium.launch(headless=HEADLESS, slow_mo=SLOW_MO)
    page = await browser.new_page()
    url = BASE_URL + str(DEPARTMENT_ID)
    print(f"Navigating to URL: {url}")
    try:
        await navigate(page, url, READY_SELECTOR)

        print("\n── Municipio ──────────────────────────────────")
        municipio = await select_option(page, "seleccione el municipio", 0)
        print(f"  ✓ Municipio selected: {municipio}")

        print("\n── Zona ───────────────────────────────────────")
        zona = await select_option(page, "seleccione la zona", 0)
        print(f"  ✓ Zona selected: {zona}")

        print("\n── Puesto ─────────────────────────────────────")
        puesto = await select_option(page, "seleccione el puesto", 0)
        print(f"  ✓ Puesto selected: {puesto}")

        await click_consultar(page)
        
        #select maximum number of mesas per page
        await select_page_size(page, size=96)
        mesas = await extract_mesas(page)
        print(f"\nDone — {len(mesas)} mesa(s) extracted.")
        
        for i, mesa in enumerate(mesas):
            await download_acta_direct(page, RAW_DATA_DIR, i)
            print(f"  ✓ Acta downloaded for Mesa {i + 1}")
        
    finally:
        await page.close()
        await browser.close()

Navigating to URL: https://divulgacione14presidente.registraduria.gov.co/departamento/72
Opening: https://divulgacione14presidente.registraduria.gov.co/departamento/72
Page loaded.

── Municipio ──────────────────────────────────
Clicking 'seleccione el municipio' container...
  Dropdown is open!
  Found 4 option(s):
    [0] 006 — CUMARIBO (100%)
    [1] 008 — LA PRIMAVERA (100%)
    [2] 001 — PUERTO CARREÑO (100%)
    [3] 002 — SANTA ROSALIA (100%)
  Selecting [0]: '006 — CUMARIBO (100%)'
  ✓ Municipio selected: 006 — CUMARIBO (100%)

── Zona ───────────────────────────────────────
Clicking 'seleccione la zona' container...
  Dropdown is open!
  Found 2 option(s):
    [0] Zona 00
    [1] Zona 99
  Selecting [0]: 'Zona 00'
  ✓ Zona selected: Zona 00

── Puesto ─────────────────────────────────────
Clicking 'seleccione el puesto' container...
  Dropdown is open!
  Found 1 option(s):
    [0] 00 - PUESTO CABECERA MUNICIPAL
  Selecting [0]: '00 - PUESTO CABECERA MUNICIPAL'
  ✓ Puesto selec

### Download All the Actas from a Municipio

This is the list of all the Municipios and their websites:

Website     Municipio
---
https://divulgacione14presidente.registraduria.gov.co/departamento/60		AMAZONAS 
https://divulgacione14presidente.registraduria.gov.co/departamento/01		ANTIOQUIA 
https://divulgacione14presidente.registraduria.gov.co/departamento/40		ARAUCA 
https://divulgacione14presidente.registraduria.gov.co/departamento/03		ATLANTICO
https://divulgacione14presidente.registraduria.gov.co/departamento/16		BOGOTA D.C.
https://divulgacione14presidente.registraduria.gov.co/departamento/05		BOLIVAR
https://divulgacione14presidente.registraduria.gov.co/departamento/07		BOYACA 
https://divulgacione14presidente.registraduria.gov.co/departamento/09		CALDAS
https://divulgacione14presidente.registraduria.gov.co/departamento/44		CAQUETA 
https://divulgacione14presidente.registraduria.gov.co/departamento/46		CASANARE
https://divulgacione14presidente.registraduria.gov.co/departamento/11		CAUCA 
https://divulgacione14presidente.registraduria.gov.co/departamento/12		CESAR 
https://divulgacione14presidente.registraduria.gov.co/departamento/17		CHOCO
https://divulgacione14presidente.registraduria.gov.co/departamento/88		CONSULADOS
https://divulgacione14presidente.registraduria.gov.co/departamento/13		CORDOBA
https://divulgacione14presidente.registraduria.gov.co/departamento/15		CUNDINAMARCA 
https://divulgacione14presidente.registraduria.gov.co/departamento/50		GUAINIA 
https://divulgacione14presidente.registraduria.gov.co/departamento/54		GUAVIARE 
https://divulgacione14presidente.registraduria.gov.co/departamento/19		HUILA
https://divulgacione14presidente.registraduria.gov.co/departamento/48		LA GUAJIRA 
https://divulgacione14presidente.registraduria.gov.co/departamento/21		MAGDALENA
https://divulgacione14presidente.registraduria.gov.co/departamento/52		META
https://divulgacione14presidente.registraduria.gov.co/departamento/23		NARIÑO
https://divulgacione14presidente.registraduria.gov.co/departamento/25		NORTE DE SAN
https://divulgacione14presidente.registraduria.gov.co/departamento/64		PUTUMAYO
https://divulgacione14presidente.registraduria.gov.co/departamento/26		QUINDIO 
https://divulgacione14presidente.registraduria.gov.co/departamento/24		RISARALDA 
https://divulgacione14presidente.registraduria.gov.co/departamento/56		SAN ANDRES
https://divulgacione14presidente.registraduria.gov.co/departamento/27		SANTANDER
https://divulgacione14presidente.registraduria.gov.co/departamento/28		SUCRE
https://divulgacione14presidente.registraduria.gov.co/departamento/29		TOLIMA
https://divulgacione14presidente.registraduria.gov.co/departamento/31		VALLE
https://divulgacione14presidente.registraduria.gov.co/departamento/68		VAUPES
https://divulgacione14presidente.registraduria.gov.co/departamento/72		VICHADA

In [8]:
async with async_playwright() as pw:
    browser = await pw.chromium.launch(headless=HEADLESS, slow_mo=SLOW_MO)
    page = await browser.new_page()
    url = BASE_URL + str(DEPARTMENT_ID)
    print(f"Navigating to URL: {url}")
    try:
        await navigate(page, url, READY_SELECTOR)
        # download all actas
        await download_all_actas(page, RAW_DATA_DIR)
    except Exception as e:
        print(f"An error occurred: {e}")
    finally:
        await browser.close()
        

Navigating to URL: https://divulgacione14presidente.registraduria.gov.co/departamento/60
Opening: https://divulgacione14presidente.registraduria.gov.co/departamento/60
Page loaded.
2026-06-21 16:43:17.212 | INFO     | elecc_colombia.actas_scraper:download_all_actas:137 - Found 11 municipio(s).
2026-06-21 16:43:21.140 | INFO     | elecc_colombia.actas_scraper:download_all_actas:142 -   [010 — EL ENCANTO (100%)] 1 zona(s).
2026-06-21 16:43:25.073 | INFO     | elecc_colombia.actas_scraper:download_all_actas:147 -     [Zona 00] 1 puesto(s).

── Consultar ──────────────────────────────────
  Clicking Consultar...
  Results loaded.

── Mesas (3 found) ────────────────────────
  [✓] Mesa 1
  [✓] Mesa 2
  [✓] Mesa 3
2026-06-21 16:43:28.860 | INFO     | elecc_colombia.actas_scraper:download_all_actas:163 -       Downloading 010_EL_ENCANTO_ZONA_00_00_CORREGIMIENTO_DEPARTAMENTAL_mesa_001.pdf...
2026-06-21 16:43:28.860 | DEBUG    | elecc_colombia.actas_scraper:download_acta_direct:113 -   Clicking